# Code-aware retrieval

Code-aware retrieval follows relationships written inside source code. It does
not guess file roles from folder names. Python and MATLAB files may be stored
in organized repositories or together in one flat folder.


In [13]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


QAChat root: /Users/yuwei.chang/OnDevice/GitHub/QAChat


In [14]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [15]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


QAChat environment ready: /Users/yuwei.chang/OnDevice/GitHub/QAChat/.venv/bin/python


## 1. Reuse natural source units

The source extensions from tutorial 50 already identify files, classes,
functions, and methods. The graph reuses those units rather than finding the
same entities again.


In [16]:
from qachat.repository import scan_repository

report = scan_repository(
    root,
    extensions=[".py", ".m"],
    include_paths=["qachat"],
)

print(f"Selected files: {len(report['selected_files'])}")


Selected files: 27


In [17]:
from qachat.sources.sources import (
    extract_source_units,
    read_repository_files,
)

source_files = read_repository_files(report)
units = extract_source_units(source_files)

print(f"Natural units: {len(units)}")


Natural units: 125


In [18]:
def count_words(text: str) -> int:
    return len(text.split())


In [19]:
from qachat.chunking import chunk_source_units

chunks = chunk_source_units(
    units,
    count_tokens=count_words,
    target_tokens=300,
    max_tokens=400,
    overlap_tokens=40,
)

print(f"Searchable chunks: {len(chunks)}")


Searchable chunks: 144


## 2. Build a code graph

The shared graph creates entities and `contains` relationships from natural
units. Small language extensions add calls, imports, and inheritance.


In [20]:
from qachat.retrievals import build_code_graph

code_graph = build_code_graph(source_files, units)

print(f"Entities: {len(code_graph.entities)}")
print(f"Relationships: {len(code_graph.relations)}")


Entities: 196
Relationships: 980


In [21]:
for relation in code_graph.relations[:10]:
    print(
        f"{relation.source} "
        f"--{relation.relation}--> "
        f"{relation.target}"
    )


qachat.chat --contains--> qachat.chat.ChatSession
qachat.chat --contains--> qachat.chat._check_bool
qachat.chat --contains--> qachat.chat._check_max_new_tokens
qachat.chat --contains--> qachat.chat._check_system_prompt
qachat.chat --contains--> qachat.chat._check_temperature
qachat.chat --imports--> __future__.annotations
qachat.chat --imports--> gc
qachat.chat --imports--> huggingface_hub.snapshot_download
qachat.chat --imports--> importlib.import_module
qachat.chat --imports--> mlx.nn.Module


We can also choose one specific entity and inspect its immediate
relationships:

- **Incoming** relationships show which code points to or contains the entity.
- **Outgoing** relationships show which code the entity points to or uses.

Here we inspect `qachat.chat._check_temperature`.

In [29]:
target = "qachat.chat._check_temperature"
print(target)

print("\nIncoming:")
for relation in code_graph.relations:
    if relation.target == target:
        print(
            f"  {relation.source} "
            f"--{relation.relation}--> {target}"
        )

print("\nOutgoing:")
for relation in code_graph.relations:
    if relation.source == target:
        print(
            f"  {target} "
            f"--{relation.relation}--> {relation.target}"
        )

qachat.chat._check_temperature

Incoming:
  qachat.chat --contains--> qachat.chat._check_temperature
  qachat.chat.ChatSession.__init__ --calls--> qachat.chat._check_temperature

Outgoing:
  qachat.chat._check_temperature --calls--> ValueError
  qachat.chat._check_temperature --calls--> float
  qachat.chat._check_temperature --calls--> isinstance


## 3. Resolve an explicit symbol

This step is deterministic. The question must name a code entity; no LLM
guesses which entity exists.


In [30]:
from qachat.retrievals import resolve_code_entities

QUESTION = "Which code is related to semantic_search?"
matches = resolve_code_entities(code_graph, QUESTION)

for match in matches:
    print(f"{match.kind}: {match.name}")


python_function: qachat.retrievals.strategies.retrieval_semantic.semantic_search


## 4. Retrieve its code relationships

Distance 0 is the named entity. Distance 1 is directly connected through a
call, import, inheritance, or containment relationship.


In [31]:
from qachat.retrievals import code_search

results = code_search(
    code_graph,
    chunks,
    QUESTION,
    count=5,
    max_distance=1,
)

for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. distance={result.distance} "
        f"symbol={result.chunk.symbol}"
    )


1. distance=0 symbol=semantic_search
2. distance=1 symbol=__init__
3. distance=1 symbol=_cosine_similarity
4. distance=1 symbol=SemanticResult
5. distance=1 symbol=retrieval_semantic


In [32]:
for result in results:
    print(f"\n>>> {result.entity} <<<\n")
    print(result.chunk.text)



>>> qachat.retrievals.strategies.retrieval_semantic.semantic_search <<<

def semantic_search(
    index: SemanticIndex,
    query_text: str,
    embedding_session: EmbeddingSession,
    *,
    count: int = 5,
) -> list[SemanticResult]:
    """Rank indexed chunks by cosine similarity to one query.

    Parameters
    ----------
    index: SemanticIndex
        Precomputed repository semantic index.
    query_text: str
        Query text already formatted for the selected embedding model.
    embedding_session: EmbeddingSession
        Loaded embedding model used to make the query vector.
    count: int, optional
        Maximum number of ranked results to return.

    Returns
    -------
    list[SemanticResult]
        Highest-scoring chunks first.

    Raises
    ------
    ValueError
        If the index is empty, count is invalid, or dimensions disagree.

    """

    if not index.chunks or not index.embeddings:
        raise ValueError("The semantic index is empty.")

    if len(i

## Result

QAChat reused natural source units, added Python-specific or MATLAB-specific
relationships, and retrieved code connected to an explicitly named symbol.
